# 프레임워크별 Tool 만들기 비교 실습

PydanticAI, LangChain, LangGraph, Deep Agents 4가지 프레임워크에서
**같은 도구(날씨 조회 + 덧셈 계산기)** 를 만들어보고 차이를 직접 체감합니다.

| 섹션 | 프레임워크 | 핵심 패턴 |
|------|-----------|----------|
| 1 | PydanticAI | `@agent.tool_plain`, `@agent.tool` |
| 2 | LangChain | `@tool`, `@tool(args_schema=...)`, `ToolRuntime` |
| 3 | LangGraph | LangChain 도구를 그래프 노드에서 사용 |
| 4 | Deep Agents | 일반 Python 함수를 리스트로 전달 |
| 5 | **언제 뭘 쓰냐?** | 프레임워크별 적합한 상황 + 선택 플로우차트 |
| 6 | 최종 비교 정리 | 한눈에 보는 비교표 |

## 0. 공통 환경 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

# 모든 프레임워크에서 공통으로 쓸 Mock 날씨 데이터
WEATHER_DB = {
    "서울": "맑음, 22°C, 습도 45%",
    "부산": "흐림, 19°C, 오후 비 예보",
    "제주": "맑음, 24°C, 바람 강함",
    "Seoul": "Clear, 22°C",
    "Tokyo": "Cloudy, 18°C",
}

print("환경 설정 완료")

---
## 1. PydanticAI — `@agent.tool_plain` / `@agent.tool`

도구를 **agent 인스턴스에 직접 바인딩**하는 방식입니다.

- `@agent.tool_plain` : 컨텍스트 없이 동작하는 순수 함수
- `@agent.tool` : `RunContext`로 대화 상태에 접근 가능

In [ ]:
from pydantic_ai import Agent

pai_agent = Agent(
    "openai:gpt-5.4",
    instructions="도구를 사용해서 답변하라. 한국어로 답변.",
)


# --- tool_plain: 컨텍스트 필요 없는 순수 함수 ---
@pai_agent.tool_plain
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회한다."""
    result = WEATHER_DB.get(city, f"{city}: 데이터 없음")
    print(f"  [PydanticAI Tool] get_weather('{city}') → {result}")
    return result


@pai_agent.tool_plain
def add(a: int, b: int) -> int:
    """두 숫자를 더한다."""
    print(f"  [PydanticAI Tool] add({a}, {b}) → {a + b}")
    return a + b


print("PydanticAI agent + 도구 등록 완료")

In [ ]:
# 실행
result = await pai_agent.run("서울 날씨 알려주고, 17 + 28도 계산해줘.")
print(f"\n최종 답변: {result.output}")

### PydanticAI 포인트 정리

```
@agent.tool_plain    →  순수 함수 (컨텍스트 X)
@agent.tool          →  RunContext로 대화 상태 접근 가능
```
- 데코레이터를 붙이는 순간 해당 agent에 종속됨
- 다른 agent에서 같은 도구를 쓰려면 다시 등록해야 함

---
## 2. LangChain — `@tool` 데코레이터

도구를 **독립적으로** 만들고, 나중에 agent에 연결하는 방식입니다.

- `@tool` : 기본 도구 생성
- `@tool(args_schema=...)` : Pydantic 모델로 복잡한 스키마 정의
- `ToolRuntime` : 대화 상태 접근

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

lc_model = ChatOpenAI(model="gpt-5.4")


# --- 방법 A: 기본 @tool ---
@tool
def lc_get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    result = WEATHER_DB.get(city, f"{city}: 데이터 없음")
    print(f"  [LangChain Tool] get_weather('{city}') → {result}")
    return result


@tool
def lc_add(a: int, b: int) -> int:
    """두 숫자를 더합니다."""
    print(f"  [LangChain Tool] add({a}, {b}) → {a + b}")
    return a + b


# 도구 스키마 확인
print("도구 이름:", lc_get_weather.name)
print("입력 스키마:", lc_get_weather.args_schema.model_json_schema())

In [ ]:
# --- 방법 B: args_schema로 복잡한 입력 정의 ---
from pydantic import BaseModel, Field


class WeatherQuery(BaseModel):
    """날씨 조회 파라미터."""
    city: str = Field(description="조회할 도시 이름")
    unit: str = Field(default="celsius", description="온도 단위: celsius 또는 fahrenheit")


@tool(args_schema=WeatherQuery)
def lc_get_weather_v2(city: str, unit: str = "celsius") -> str:
    """도시의 현재 날씨를 지정한 단위로 조회합니다."""
    result = WEATHER_DB.get(city, f"{city}: 데이터 없음")
    print(f"  [LangChain Tool v2] get_weather('{city}', unit='{unit}') → {result}")
    return result


print("복합 스키마:", lc_get_weather_v2.args_schema.model_json_schema())

In [ ]:
# agent에 도구를 연결하고 실행
lc_agent = create_agent(
    model=lc_model,
    tools=[lc_get_weather, lc_add],
    system_prompt="도구를 사용해서 답변하세요. 한국어로 답변.",
)

result = lc_agent.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨 알려주고, 17 + 28도 계산해줘."}]}
)
print(f"\n최종 답변: {result['messages'][-1].content}")

### LangChain 포인트 정리

```
@tool                       →  기본 도구 (함수 독립 생성)
@tool(args_schema=Model)    →  Pydantic으로 복잡한 스키마
ToolRuntime                 →  대화 상태 접근
create_agent(tools=[...])   →  agent에 연결
```
- 도구가 agent와 독립적 → 여러 agent에서 재사용 가능
- `args_schema`로 Field(description=...)까지 세밀하게 정의 가능

---
## 3. LangGraph — LangChain 도구를 그래프 노드에서 사용

LangGraph는 자체 도구 문법이 없습니다.
LangChain `@tool`로 만든 도구를 **그래프 노드 안의 agent**에서 활용합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent
from typing import TypedDict


# 그래프 상태 정의
class GraphState(TypedDict):
    question: str
    answer: str


# 노드 함수: 내부에서 LangChain 도구가 달린 agent를 사용
def tool_node(state: GraphState) -> dict:
    # 위에서 만든 LangChain 도구를 그대로 재사용!
    agent = create_agent(
        model=lc_model,
        tools=[lc_get_weather, lc_add],
        system_prompt="도구를 사용해서 답변하세요. 한국어로 답변.",
    )
    result = agent.invoke(
        {"messages": [{"role": "user", "content": state["question"]}]}
    )
    return {"answer": result["messages"][-1].content}


# 그래프 빌드
builder = StateGraph(GraphState)
builder.add_node("tool_agent", tool_node)
builder.add_edge(START, "tool_agent")
builder.add_edge("tool_agent", END)
graph = builder.compile()

print("LangGraph 빌드 완료")

In [ ]:
# 실행
result = graph.invoke({"question": "부산 날씨 알려주고, 100 + 250도 계산해줘.", "answer": ""})
print(f"\n최종 답변: {result['answer']}")

### LangGraph 포인트 정리

```
도구 생성  →  LangChain @tool 그대로 사용
도구 연결  →  그래프 노드 안에서 create_agent(tools=[...])로 사용
LangGraph의 역할  →  워크플로우(노드 → 엣지) 오케스트레이션
```
- LangGraph 자체는 도구 생성 문법이 없음
- LangChain 도구를 재사용하므로 별도 학습 비용 없음
- 복잡한 분기/루프 워크플로우에 강점

---
## 4. Deep Agents — 일반 Python 함수

**데코레이터 없이** 일반 함수를 리스트로 넘기면 도구가 됩니다.
가장 간단한 방식입니다.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend


# 그냥 일반 Python 함수! 데코레이터 없음!
def da_get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    result = WEATHER_DB.get(city, f"{city}: 데이터 없음")
    print(f"  [Deep Agents Tool] get_weather('{city}') → {result}")
    return result


def da_add(a: int, b: int) -> int:
    """두 숫자를 더합니다."""
    print(f"  [Deep Agents Tool] add({a}, {b}) → {a + b}")
    return a + b


da_agent = create_deep_agent(
    model=lc_model,
    tools=[da_get_weather, da_add],
    system_prompt="도구를 사용해서 답변하세요. 한국어로 답변.",
    backend=LocalShellBackend("./", virtual_mode=True),
)

print("Deep Agents agent 생성 완료")

In [ ]:
# 실행
result = da_agent.invoke(
    {"messages": [{"role": "user", "content": "제주 날씨 알려주고, 50 + 75도 계산해줘."}]}
)
print(f"\n최종 답변: {result['messages'][-1].content}")

### Deep Agents 포인트 정리

```
도구 생성  →  일반 Python 함수 (데코레이터 없음)
도구 연결  →  create_deep_agent(tools=[func1, func2])
스키마     →  타입힌트 + docstring에서 자동 생성
```
- 가장 간단 (데코레이터 불필요)
- `ls`, `read_file`, `write_file`, `grep`, `execute` 등 빌트인 도구 자동 포함
- 코딩/파일 작업에 특화된 프레임워크

---
## 6. 최종 비교 정리

| | PydanticAI | LangChain | LangGraph | Deep Agents |
|---|---|---|---|---|
| **데코레이터** | `@agent.tool_plain` | `@tool` | 없음 (LangChain 사용) | 없음 |
| **등록 방식** | agent에 직접 바인딩 | `create_agent(tools=[])` | 노드 안에서 사용 | `create_deep_agent(tools=[])` |
| **복잡한 스키마** | 타입힌트 | `args_schema=Pydantic` | LangChain과 동일 | 타입힌트 |
| **컨텍스트 접근** | `RunContext` | `ToolRuntime` | State를 노드가 관리 | 자동 (빌트인) |
| **재사용성** | agent 종속 | 독립적, 높음 | LangChain 도구 공유 | 독립적, 높음 |
| **빌트인 도구** | 없음 | 없음 | 없음 | 파일/쉘 등 자동 |

### 핵심 공통점

4개 프레임워크 모두 결국 같은 원리:

```
함수 이름 + docstring + 타입힌트  →  JSON 스키마로 변환  →  LLM에게 전달  →  LLM이 호출 판단
```

차이는 "어떻게 감싸서 agent에 연결하느냐"의 문법 차이일 뿐입니다.

---
## 5. 최종 비교 정리

| | PydanticAI | LangChain | LangGraph | Deep Agents |
|---|---|---|---|---|
| **데코레이터** | `@agent.tool_plain` | `@tool` | 없음 (LangChain 사용) | 없음 |
| **등록 방식** | agent에 직접 바인딩 | `create_agent(tools=[])` | 노드 안에서 사용 | `create_deep_agent(tools=[])` |
| **복잡한 스키마** | 타입힌트 | `args_schema=Pydantic` | LangChain과 동일 | 타입힌트 |
| **컨텍스트 접근** | `RunContext` | `ToolRuntime` | State를 노드가 관리 | 자동 (빌트인) |
| **재사용성** | agent 종속 | 독립적, 높음 | LangChain 도구 공유 | 독립적, 높음 |
| **빌트인 도구** | 없음 | 없음 | 없음 | 파일/쉘 등 자동 |

### 핵심 공통점

4개 프레임워크 모두 결국 같은 원리:

```
함수 이름 + docstring + 타입힌트  →  JSON 스키마로 변환  →  LLM에게 전달  →  LLM이 호출 판단
```

차이는 "어떻게 감싸서 agent에 연결하느냐"의 문법 차이일 뿐입니다.